<a href="https://colab.research.google.com/github/ankit-rathi/AR-Talks/blob/master/dbt_from_zero_to_advanced_duckdb_colab_worldclass_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# dbt from Zero to Advanced — World-Class Colab Lab
### A theory-first, hands-on analytics engineering course with dbt + DuckDB

**Environment:** Google Colab  
**Database:** DuckDB  
**dbt line:** dbt v1.12 + `dbt-duckdb` 1.11.x  
**Teaching loop:** **Explain → Implement → Inspect → Experiment → Break → Debug → Measure → Design**

---

## What you will build

We will build **Acme Mart**, a small but realistic e-commerce analytics project.

```text
raw data
   │
   ├── sources / seeds
   ▼
staging models
   │
   ▼
intermediate models
   │
   ▼
marts
   │
   ├── tests
   ├── documentation
   ├── lineage
   ├── macros / Jinja
   ├── incremental models
   ├── snapshots
   ├── contracts
   ├── unit tests
   ├── exposures
   └── artifacts
```

The objective is not to memorize commands. The objective is to understand:

> **What is this feature? Why does it exist? How does it work? When should I use it? What can go wrong? How would I use it in production?**

### Learning contract

Every major feature follows:

1. **Theory**
2. **Minimal example**
3. **Hands-on**
4. **Inspect**
5. **Break / debug where useful**
6. **Production thinking**
7. **Exercise**

> **Version note:** this notebook intentionally targets the dbt v1.12 line because it is the current dbt v1 line at refresh time. dbt v2 is a newer engine line with different installation and compatibility considerations.

## Table of contents

### Foundations
1. The analytics engineering problem
2. dbt mental model
3. dbt project anatomy
4. Reproducible Colab environment
5. DuckDB + dbt concurrency: the failure we are fixing

### Core dbt workflow
6. Your first model
7. Materializations
8. `ref()` and the DAG
9. Seeds
10. Sources and `source()`
11. Staging → intermediate → marts
12. `dbt build`

### Quality engineering
13. Data tests
14. Singular tests
15. Generic tests
16. Jinja and compilation
17. Macros
18. Variables and environment variables
19. Configs, tags and metadata
20. Unit tests
21. Contracts and structural guarantees

### Performance and history
22. Incremental models
23. Incremental strategies and late-arriving data
24. Snapshots / slowly changing history
25. Source freshness

### Governance and observability
26. Documentation
27. Exposures
28. Artifacts and metadata
29. Selection and graph traversal
30. Debugging methodology
31. Hooks and analyses

### Reuse and advanced architecture
32. Packages / `dbt_utils`
33. Python models with dbt-duckdb
34. Model versions
35. State-aware CI/CD concepts
36. Production project architecture
37. Capstone
38. Final knowledge map

# 1. The analytics engineering problem

A growing transformation system needs more than SQL.

Without a transformation framework:

```text
database tables
      │
      ├── SQL script A
      ├── SQL script B
      ├── SQL script C
      ├── undocumented assumptions
      └── manual execution order
```

The difficult questions are:

- Which model depends on this source?
- What must run first?
- Which assumptions are tested?
- What does each column mean?
- Which dashboard uses this model?
- Can I change it safely?
- Can I rebuild only the affected graph?
- How do I reuse transformation logic?
- Is the upstream data fresh?

## The analytics engineering idea

dbt applies software-engineering practices to the **transformation layer**:

```text
SQL transformations
       +
dependency graph
       +
testing
       +
documentation
       +
version control
       +
reusable code
       +
metadata / artifacts
       =
maintainable analytics system
```

dbt does not replace the database.

> **You write transformation logic. The database executes SQL. dbt coordinates, compiles, tests, documents, and manages the transformation project.**

# 2. The dbt mental model

A useful first mental model is:

```text
             upstream systems
                    │
                    ▼
             ┌─────────────┐
             │   sources   │
             └──────┬──────┘
                    │
                    ▼
             ┌─────────────┐
             │   staging   │  clean / standardize
             └──────┬──────┘
                    │
                    ▼
             ┌─────────────┐
             │ intermediate│  reusable business logic
             └──────┬──────┘
                    │
                    ▼
             ┌─────────────┐
             │    marts    │  consumer-facing models
             └──────┬──────┘
                    │
          ┌─────────┼─────────┐
          ▼         ▼         ▼
      dashboard   ML/AI    applications
```

Around this graph dbt adds:

```text
tests + documentation + lineage + Jinja + macros
+ configs + snapshots + incremental processing + artifacts
```

## The most important dbt function

```sql
{{ ref('stg_orders') }}
```

`ref()` does two jobs:

1. resolves the relation name for the current environment;
2. creates a dependency edge in the dbt graph.

That second property turns a collection of SQL files into a **DAG-aware project**.

Reference: https://docs.getdbt.com/reference/dbt-jinja-functions/ref

# 3. Project anatomy

A dbt project is more than SQL files.

```text
dbt_project.yml       ← project configuration
profiles.yml          ← connection profile

models/
  staging/
  intermediate/
  marts/
  *.yml               ← documentation/tests/configuration

seeds/                ← small CSV reference data
snapshots/            ← historical record tracking
tests/                ← singular SQL tests
macros/               ← reusable Jinja functions
analyses/             ← version-controlled analytical SQL
packages.yml          ← external dependencies

target/               ← generated artifacts / compiled output
```

### Layer responsibilities

| Layer | Typical responsibility | Typical materialization |
|---|---|---|
| staging | rename, cast, standardize | view |
| intermediate | joins and reusable business logic | view / ephemeral / table |
| marts | business-facing facts/dimensions | table / incremental |

The folders are conventions. What matters is the combination of location, configuration, naming, dependencies, and team design standards.

# 4. Reproducible Colab environment

The previous notebook installed dbt directly into Colab's global Python environment. That produced unrelated dependency warnings such as protobuf conflicts.

The dbt installation could still work, but changing the global environment makes a reusable teaching notebook fragile.

## Better pattern

Use an isolated virtual environment:

```text
Colab Python
    │
    ├── normal notebook libraries
    │
    └── /content/dbt_venv
            ├── dbt-core
            ├── dbt-duckdb
            └── adapter dependencies
```

This makes the lab more reproducible and avoids coupling dbt's dependency set to unrelated Colab packages.

In [ ]:
# Create an isolated environment for dbt.
# This prevents dbt dependencies from modifying Colab's global Python environment.

from pathlib import Path
import os
import subprocess
import sys

VENV = Path("/content/dbt_venv")
if not VENV.exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)

DBT_BIN = VENV / "bin" / "dbt"
PYTHON_BIN = VENV / "bin" / "python"

# Pin the versions used by this refreshed lab.
subprocess.run(
    [
        str(PYTHON_BIN), "-m", "pip", "install", "-q",
        "dbt-core==1.12.5",
        "dbt-duckdb==1.11.0",
        "pandas",
    ],
    check=True,
)

print("dbt:", DBT_BIN)
print("venv Python:", PYTHON_BIN)

In [ ]:
# Verify the exact dbt and adapter versions.
# Version verification is part of reproducibility, not just housekeeping.

result = subprocess.run(
    [str(DBT_BIN), "--version"],
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)

# 5. The failure we are fixing: DuckDB file locking

### What happened in the previous notebook?

It kept a Python DuckDB connection alive while dbt attempted to write the same file:

```python
con = duckdb.connect("acme_mart.duckdb")

# connection remains open
!dbt run
```

The captured failure was:

```text
IO Error: Could not set lock on file
"/content/dbt_colab_lab/acme_mart.duckdb":
Conflicting lock is held in /usr/bin/python3.13
```

This is a **resource-lifecycle problem**, not a SQL problem.

### Correct pattern

```text
Python query
   ↓
open connection
   ↓
read
   ↓
close connection
   ↓
dbt command
   ↓
dbt opens database
   ↓
build
   ↓
close
```

> **Never keep a file-backed DuckDB write connection open across a dbt command.**

DuckDB's concurrency documentation explains that file-backed read/write access uses file locking and that multi-process writes require care.

Reference: https://duckdb.org/docs/current/connect/concurrency

In [ ]:
# Create the project skeleton and helper functions.
# The database helper deliberately opens and closes a connection for each query.

from pathlib import Path
import os
import shutil
import subprocess
import json
import pandas as pd

PROJECT = Path("/content/dbt_colab_lab")
if PROJECT.exists():
    shutil.rmtree(PROJECT)

for folder in [
    "models/staging",
    "models/intermediate",
    "models/marts",
    "seeds",
    "snapshots",
    "tests",
    "macros",
    "analyses",
]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

os.environ["DBT_PROFILES_DIR"] = str(PROJECT)
os.chdir(PROJECT)

def run_dbt(*args, check=True):
    """Run dbt as a subprocess and return its completed process."""
    cmd = [str(DBT_BIN), *map(str, args)]
    print("$", " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=PROJECT,
        env=os.environ.copy(),
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"dbt command failed: {result.returncode}")
    return result

def duck_query(sql, read_only=True):
    """Run one DuckDB query and immediately close the connection."""
    import duckdb
    con = duckdb.connect(
        str(PROJECT / "acme_mart.duckdb"),
        read_only=read_only,
    )
    try:
        return con.sql(sql).df()
    finally:
        con.close()

print("Project:", PROJECT)

In [ ]:
# Write the dbt profile and project configuration.
# The configured directory hierarchy matches the actual model folders,
# so dbt does not report unused model configuration paths.

(PROJECT / "profiles.yml").write_text("""
dbt_colab_lab:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: /content/dbt_colab_lab/acme_mart.duckdb
      threads: 1
""".strip() + "\n")

(PROJECT / "dbt_project.yml").write_text("""
name: dbt_colab_lab
version: "1.0.0"
config-version: 2
profile: dbt_colab_lab

model-paths: ["models"]
seed-paths: ["seeds"]
snapshot-paths: ["snapshots"]
test-paths: ["tests"]
macro-paths: ["macros"]
analysis-paths: ["analyses"]

vars:
  minimum_order_amount: 700

models:
  dbt_colab_lab:
    staging:
      +materialized: view
      +tags: ["staging"]
    intermediate:
      +materialized: view
      +tags: ["intermediate"]
    marts:
      +materialized: table
      +tags: ["mart"]
""".strip() + "\n")

run_dbt("debug")

# 6. Your first model

A dbt model is usually a SQL file whose final statement is a `SELECT`.

You write:

```sql
select 1 as id, 'hello dbt' as message
```

dbt determines how that SQL becomes a database relation.

```text
your SQL
   ↓
dbt compilation
   ↓
adapter-specific execution
   ↓
DuckDB
   ↓
view / table / incremental relation
```

**dbt** handles project parsing, compilation, dependency ordering, execution, testing and metadata.

**DuckDB** handles SQL execution and storage.

In [ ]:
# Create the first model in models/staging so the directory-level configuration applies.

(PROJECT / "models/staging/hello.sql").write_text("""
select
    1 as id,
    'hello dbt' as message
""".strip() + "\n")

run_dbt("run", "--select", "hello")
print(duck_query("select * from hello"))

# 7. Materializations

A **materialization** answers:

> "How should dbt persist this model?"

### View

```text
model SQL → database VIEW
```

Low storage; query work occurs when consumers query the view.

### Table

```text
model SQL → physical TABLE
```

The transformation runs during the dbt job and the result is stored.

### Incremental

```text
first run  → build full table
later runs → process selected new/changed rows
```

### Ephemeral

```text
model SQL → inlined into downstream SQL
```

No persistent relation is created.

### Decision framework

Ask:

1. How expensive is the transformation?
2. How frequently is it queried?
3. Is it reused by many downstream models?
4. Is storage cheaper than repeated compute?
5. Is the dataset large enough to justify incremental processing?
6. Does it need to be directly queryable?

There is no universally best materialization.

In [ ]:
# Create examples of view, table, and ephemeral materializations.
# The SQL is intentionally identical in complexity so the learner can focus on persistence.

(PROJECT / "models/intermediate/demo_view.sql").write_text("""
{{ config(materialized='view') }}
select 1 as id, 'view' as materialization
""".strip() + "\n")

(PROJECT / "models/marts/demo_table.sql").write_text("""
{{ config(materialized='table') }}
select 1 as id, 'table' as materialization
""".strip() + "\n")

(PROJECT / "models/intermediate/demo_ephemeral.sql").write_text("""
{{ config(materialized='ephemeral') }}
select 1 as id, 'ephemeral' as materialization
""".strip() + "\n")

run_dbt("run", "--select", "demo_view demo_table demo_ephemeral")
print(duck_query("""
select table_name, table_type
from information_schema.tables
where table_schema = 'main'
  and table_name like 'demo_%'
order by table_name
"""))

# 8. `ref()` and the DAG

Suppose:

```text
stg_customers ─────┐
                   ▼
             customer_orders
                   ▲
stg_orders ────────┘
```

If the downstream SQL says:

```sql
select *
from {{ ref('stg_orders') }}
```

dbt records:

```text
stg_orders → customer_orders
```

### Why not hard-code table names?

Avoid:

```sql
from main.stg_orders
```

Prefer:

```sql
from {{ ref('stg_orders') }}
```

because `ref()`:

- resolves the relation for the current target,
- creates a graph dependency,
- enables ordered execution,
- supports selective graph operations.

This is the conceptual heart of dbt.

# 9. Build deterministic Acme Mart raw data

We will use deterministic data so the lab is:

- credential-free,
- reproducible,
- easy to reset,
- easy to corrupt deliberately,
- independent of external APIs.

The raw domain contains:

- customers,
- orders,
- order items,
- payments,
- events.

In production, these might arrive from application databases, SaaS tools, CDC pipelines, streaming systems, or ingestion platforms.

In [ ]:
# Create deterministic raw CSV files.
# Seeds are used here for teaching; production raw data is commonly loaded upstream
# and declared as dbt sources.

customers = pd.DataFrame([
    [1, "Asha", "Delhi", "2025-01-10"],
    [2, "Rahul", "Mumbai", "2025-01-15"],
    [3, "Meera", "Bengaluru", "2025-02-03"],
    [4, "Kabir", "Pune", "2025-02-17"],
    [5, "Nisha", "Chennai", "2025-03-01"],
], columns=["customer_id", "customer_name", "city", "signup_date"])

orders = pd.DataFrame([
    [1001, 1, "2025-03-01", "completed", 1200],
    [1002, 1, "2025-03-03", "completed", 800],
    [1003, 2, "2025-03-05", "cancelled", 500],
    [1004, 2, "2025-03-08", "completed", 2200],
    [1005, 3, "2025-03-10", "completed", 1500],
    [1006, 3, "2025-03-12", "completed", 700],
    [1007, 4, "2025-03-15", "completed", 3000],
    [1008, 5, "2025-03-18", "pending", 900],
], columns=["order_id", "customer_id", "order_date", "status", "order_amount"])

order_items = pd.DataFrame([
    [1001, 101, 1, 1200],
    [1002, 102, 2, 400],
    [1003, 103, 1, 500],
    [1004, 104, 1, 2200],
    [1005, 105, 3, 500],
    [1006, 106, 1, 700],
    [1007, 107, 2, 1500],
    [1008, 108, 1, 900],
], columns=["order_id", "product_id", "quantity", "item_amount"])

payments = pd.DataFrame([
    [5001, 1001, "2025-03-01", 1200, "paid"],
    [5002, 1002, "2025-03-03", 800, "paid"],
    [5003, 1003, "2025-03-05", 500, "refunded"],
    [5004, 1004, "2025-03-08", 2200, "paid"],
    [5005, 1005, "2025-03-10", 1500, "paid"],
    [5006, 1006, "2025-03-12", 700, "paid"],
    [5007, 1007, "2025-03-15", 3000, "paid"],
    [5008, 1008, "2025-03-18", 900, "pending"],
], columns=["payment_id", "order_id", "payment_date", "amount", "payment_status"])

events = pd.DataFrame([
    [1, 1, "2025-03-01 10:00:00", "view"],
    [2, 1, "2025-03-01 10:05:00", "add_to_cart"],
    [3, 2, "2025-03-01 11:00:00", "view"],
    [4, 2, "2025-03-02 09:00:00", "purchase"],
    [5, 3, "2025-03-02 12:00:00", "view"],
], columns=["event_id", "customer_id", "event_at", "event_type"])

for name, df in {
    "raw_customers": customers,
    "raw_orders": orders,
    "raw_order_items": order_items,
    "raw_payments": payments,
    "raw_events": events,
}.items():
    df.to_csv(PROJECT / "seeds" / f"{name}.csv", index=False)

run_dbt("seed")

# 10. Seeds

A **seed** is a CSV stored with the dbt project and loaded into the database by dbt.

Good uses:

- small reference mappings,
- controlled lookup tables,
- static classifications,
- teaching data.

Poor uses:

- large transaction tables,
- high-frequency changing data,
- ingestion pipelines.

### Seed vs source

| Seed | Source |
|---|---|
| dbt owns the CSV | another system owns the data |
| dbt loads it | dbt references an existing relation |
| best for small/static data | best for upstream operational/ingested data |

In [ ]:
# Inspect the relations created by dbt seed.
# The DuckDB connection is opened only for this read and then closed.

print(duck_query("""
select table_name
from information_schema.tables
where table_schema = 'main'
  and table_name like 'raw_%'
order by table_name
"""))

# 11. Sources and `source()`

A source declaration gives dbt a named interface to an upstream table:

```yaml
sources:
  - name: acme_raw
    schema: main
    tables:
      - name: raw_orders
```

SQL then uses:

```sql
select *
from {{ source('acme_raw', 'raw_orders') }}
```

### Why sources matter

They provide:

- explicit upstream lineage,
- source documentation,
- freshness checks,
- stable names,
- a clear boundary between ingestion and transformation.

### `source()` vs `ref()`

```text
source()
  = upstream data owned outside dbt

ref()
  = another dbt-managed resource
```

In [ ]:
# Declare the raw relations as dbt sources.
# These are currently seeded by dbt only because this is a self-contained lab.

(PROJECT / "models/staging/sources.yml").write_text("""
version: 2

sources:
  - name: acme_raw
    schema: main
    tables:
      - name: raw_customers
      - name: raw_orders
      - name: raw_order_items
      - name: raw_payments
      - name: raw_events
""".strip() + "\n")

run_dbt("parse")

In [ ]:
# Build the order staging model from the declared source.
# This illustrates the production-oriented source -> staging boundary.

(PROJECT / "models/staging/stg_orders.sql").write_text("""
select
    order_id,
    customer_id,
    cast(order_date as date) as order_date,
    lower(trim(status)) as status,
    cast(order_amount as decimal(12,2)) as order_amount
from {{ source('acme_raw', 'raw_orders') }}
""".strip() + "\n")

run_dbt("run", "--select", "stg_orders")
print(duck_query("select * from stg_orders order by order_id"))

# 12. Staging → intermediate → marts

This is a design convention, not a magic dbt feature.

## Staging

Usually:

- rename,
- cast,
- standardize,
- lightly clean.

Avoid heavy business logic.

## Intermediate

Usually:

- joins,
- reusable business transformations,
- grain changes,
- shared calculations.

## Marts

Usually:

- consumer-facing business entities,
- facts,
- dimensions,
- stable analytical interfaces.

### Grain

A model should have a clear grain:

```text
stg_orders
= one row per source order

fct_orders
= one row per analytical order

customer_order_summary
= one row per customer
```

If the grain cannot be stated in one sentence, stop and clarify it.

In [ ]:
# Create the remaining staging models.
# Each is intentionally small and source-aligned.

(PROJECT / "models/staging/stg_customers.sql").write_text("""
select
    customer_id,
    trim(customer_name) as customer_name,
    trim(city) as city,
    cast(signup_date as date) as signup_date
from {{ source('acme_raw', 'raw_customers') }}
""".strip() + "\n")

(PROJECT / "models/staging/stg_order_items.sql").write_text("""
select
    order_id,
    product_id,
    quantity,
    cast(item_amount as decimal(12,2)) as item_amount
from {{ source('acme_raw', 'raw_order_items') }}
""".strip() + "\n")

(PROJECT / "models/staging/stg_payments.sql").write_text("""
select
    payment_id,
    order_id,
    cast(payment_date as date) as payment_date,
    cast(amount as decimal(12,2)) as amount,
    lower(trim(payment_status)) as payment_status
from {{ source('acme_raw', 'raw_payments') }}
""".strip() + "\n")

(PROJECT / "models/staging/stg_events.sql").write_text("""
select
    event_id,
    customer_id,
    cast(event_at as timestamp) as event_at,
    lower(trim(event_type)) as event_type
from {{ source('acme_raw', 'raw_events') }}
""".strip() + "\n")

run_dbt("run", "--select", "staging")

In [ ]:
# Build an intermediate model and a mart.
# The intermediate model owns the reusable customer/order join;
# the mart owns the business rule excluding cancelled orders.

(PROJECT / "models/intermediate/int_orders_enriched.sql").write_text("""
select
    o.order_id,
    o.order_date,
    o.customer_id,
    c.customer_name,
    c.city,
    o.status,
    o.order_amount
from {{ ref('stg_orders') }} o
left join {{ ref('stg_customers') }} c
    on o.customer_id = c.customer_id
""".strip() + "\n")

(PROJECT / "models/marts/fct_orders.sql").write_text("""
select
    order_id,
    order_date,
    customer_id,
    customer_name,
    city,
    status,
    order_amount
from {{ ref('int_orders_enriched') }}
where status <> 'cancelled'
""".strip() + "\n")

run_dbt("run", "--select", "+fct_orders")
print(duck_query("select * from fct_orders order by order_id"))

# 13. Data tests

A dbt **data test** asks:

> "Does the actual dataset satisfy this assertion?"

Common generic tests:

- `unique`
- `not_null`
- `accepted_values`
- `relationships`

Example:

```yaml
- name: order_id
  data_tests:
    - unique
    - not_null
```

Relationship:

```yaml
- name: customer_id
  data_tests:
    - relationships:
        arguments:
          to: ref('stg_customers')
          field: customer_id
```

## Test assumptions, not syntax

A useful test says something about the data contract:

```text
Every order has a valid customer.
```

rather than merely repeating the SQL structure.

Reference: https://docs.getdbt.com/docs/build/data-tests

In [ ]:
# Define documentation and core data tests.
# These tests protect keys, relationships, and controlled categorical values.

(PROJECT / "models/schema.yml").write_text("""
version: 2

models:
  - name: stg_customers
    description: "One row per source customer after light standardization."
    columns:
      - name: customer_id
        description: "Stable customer identifier."
        data_tests:
          - unique
          - not_null
      - name: customer_name
        data_tests:
          - not_null

  - name: stg_orders
    description: "One row per source order after type and status standardization."
    columns:
      - name: order_id
        data_tests:
          - unique
          - not_null
      - name: customer_id
        data_tests:
          - not_null
          - relationships:
              arguments:
                to: ref('stg_customers')
                field: customer_id
      - name: status
        data_tests:
          - accepted_values:
              arguments:
                values: ['completed', 'cancelled', 'pending']

  - name: fct_orders
    description: "One row per non-cancelled analytical order."
    columns:
      - name: order_id
        data_tests:
          - unique
          - not_null
      - name: customer_id
        data_tests:
          - relationships:
              arguments:
                to: ref('stg_customers')
                field: customer_id
""".strip() + "\n")

run_dbt("test")

## Exercise — deliberately break a relationship

Change one `customer_id` in `seeds/raw_orders.csv` to `999`.

Then run:

```text
dbt seed --select raw_orders
dbt build --select +stg_orders
```

The relationship test should expose the invalid foreign key.

Ask:

> Is the test broken, or is the data broken?

Restore the original value after the exercise.

# 14. Singular tests

A **singular test** is SQL that should return **zero rows**.

Example:

```sql
select *
from {{ ref('fct_orders') }}
where order_amount < 0
```

Pass condition:

```text
returned rows = 0
```

Failure condition:

```text
returned rows > 0
```

Use singular tests for business rules that are easier to express directly than as a reusable template.

In [ ]:
# Create a singular test for the business rule that analytical orders cannot be negative.

(PROJECT / "tests/fct_orders_no_negative_amounts.sql").write_text("""
select *
from {{ ref('fct_orders') }}
where order_amount < 0
""".strip() + "\n")

run_dbt("test", "--select", "fct_orders_no_negative_amounts")

# 15. Generic tests

A generic test is a reusable test template.

Conceptually:

```text
generic test definition
        │
        ├── model = stg_orders
        ├── column = order_amount
        ▼
generated SQL
        ▼
zero rows = PASS
```

This is analogous to a function:

```python
def positive_amount(model, column):
    ...
```

Generic tests are valuable when the same business rule is reused across many models.

In [ ]:
# Define a reusable positive_amount generic test.
# dbt supplies the model and column arguments when the test is invoked from YAML.

(PROJECT / "macros/test_positive_amount.sql").write_text("""
{% test positive_amount(model, column_name) %}

select *
from {{ model }}
where {{ column_name }} < 0

{% endtest %}
""".strip() + "\n")

(PROJECT / "models/generic_test_demo.yml").write_text("""
version: 2

models:
  - name: stg_orders
    columns:
      - name: order_amount
        data_tests:
          - positive_amount
""".strip() + "\n")

run_dbt("test", "--select", "stg_orders")

# 16. Jinja and compilation

dbt combines SQL with Jinja.

Common constructs:

```jinja
{{ expression }}

{% if condition %}
{% endif %}

{% for item in items %}
{% endfor %}
```

The critical distinction:

```text
Jinja executes during dbt compilation.
DuckDB receives the compiled SQL.
```

This means compilation is a powerful debugging tool.

Reference: https://docs.getdbt.com/docs/build/jinja-macros

In [ ]:
# Demonstrate a Jinja loop that generates ordinary SQL.
# The learner should inspect the compiled result next.

(PROJECT / "models/marts/customer_order_summary.sql").write_text("""
{% set valid_statuses = ['completed', 'pending'] %}

select
    customer_id,
    count(*) as order_count,
    sum(
        case
            when status in (
                {% for status in valid_statuses %}
                '{{ status }}'{% if not loop.last %}, {% endif %}
                {% endfor %}
            )
            then order_amount
            else 0
        end
    ) as relevant_order_value
from {{ ref('stg_orders') }}
group by customer_id
""".strip() + "\n")

run_dbt("run", "--select", "customer_order_summary")
run_dbt("compile", "--select", "customer_order_summary")

In [ ]:
# Read the compiled SQL so learners can compare Jinja source with database SQL.

compiled_root = PROJECT / "target" / "compiled" / "dbt_colab_lab" / "models"
matches = list(compiled_root.rglob("customer_order_summary.sql"))
for p in matches:
    print(f"--- {p} ---")
    print(p.read_text())

# 17. Macros

A macro is a reusable Jinja function.

Example:

```jinja
{% macro order_status_bucket(status_column) %}
case
    when {{ status_column }} = 'completed' then 'successful'
    when {{ status_column }} = 'pending' then 'awaiting_payment'
    when {{ status_column }} = 'cancelled' then 'cancelled'
    else 'unknown'
end
{% endmacro %}
```

Then:

```sql
{{ order_status_bucket('status') }}
```

### Macro design rule

Create a macro when it:

- removes meaningful repetition,
- standardizes a business rule,
- abstracts adapter-specific behavior,
- improves maintainability.

Do not hide simple SQL merely to make the project look sophisticated.

In [ ]:
# Create and use a reusable macro for order status classification.

(PROJECT / "macros/order_status_bucket.sql").write_text("""
{% macro order_status_bucket(status_column) %}
case
    when {{ status_column }} = 'completed' then 'successful'
    when {{ status_column }} = 'pending' then 'awaiting_payment'
    when {{ status_column }} = 'cancelled' then 'cancelled'
    else 'unknown'
end
{% endmacro %}
""".strip() + "\n")

(PROJECT / "models/marts/order_status_summary.sql").write_text("""
select
    {{ order_status_bucket('status') }} as status_bucket,
    count(*) as orders,
    sum(order_amount) as amount
from {{ ref('stg_orders') }}
group by 1
order by 1
""".strip() + "\n")

run_dbt("run", "--select", "order_status_summary")
print(duck_query("select * from order_status_summary"))

# 18. Variables and environment variables

## Project variables

```yaml
vars:
  minimum_order_amount: 700
```

Use:

```jinja
{{ var('minimum_order_amount') }}
```

Good for project-level configuration.

## Environment variables

```jinja
{{ env_var('MY_SETTING') }}
```

Useful for deployment-specific settings and secrets injected by the environment.

### Important

Do not treat project variables as a secret store. Credentials belong in secure credential management.

Reference: https://docs.getdbt.com/docs/build/environment-variables

In [ ]:
# Use a project variable so the business threshold is configurable without changing SQL.

(PROJECT / "models/marts/large_orders.sql").write_text("""
select *
from {{ ref('stg_orders') }}
where order_amount >= {{ var('minimum_order_amount') }}
""".strip() + "\n")

run_dbt("run", "--select", "large_orders")
print(duck_query("select * from large_orders order by order_id"))

In [ ]:
# Demonstrate an environment variable using a non-secret setting.
# In production, CI/CD would normally inject this value.

os.environ["DBT_LAB_ENV"] = "colab-dev"

(PROJECT / "models/marts/environment_demo.sql").write_text("""
select
    '{{ env_var("DBT_LAB_ENV", "unknown") }}' as environment_name
""".strip() + "\n")

run_dbt("run", "--select", "environment_demo")
print(duck_query("select * from environment_demo"))

# 19. Configs, tags and metadata

### Config

Controls resource behavior:

```jinja
{{ config(materialized='table') }}
```

### Tags

Useful for operational selection:

```yaml
tags: ["finance", "daily"]
```

Then:

```text
dbt build --select tag:finance
```

### Meta

A flexible metadata dictionary:

```yaml
meta:
  owner: finance-data
  criticality: high
```

Good metadata answers:

- Who owns it?
- What domain is it?
- How critical is it?
- Which SLA applies?
- Who consumes it?

Metadata should serve an operational purpose.

In [ ]:
# Add operational metadata to the main fact model.
# This is machine-readable project metadata, not merely decoration.

(PROJECT / "models/marts/fct_orders_properties.yml").write_text("""
version: 2

models:
  - name: fct_orders
    config:
      tags: ["finance", "daily", "critical"]
      meta:
        owner: "analytics-engineering"
        criticality: "high"
    description: "Curated order fact excluding cancelled orders."
""".strip() + "\n")

run_dbt("parse")
run_dbt("ls", "--select", "tag:finance")

# 20. Unit tests vs data tests

This distinction is fundamental.

## Data test

> **Does the actual dataset satisfy an assertion?**

Examples:

- key is unique,
- key is non-null,
- relationship exists,
- value belongs to an allowed set.

## Unit test

> **Does my transformation logic produce the expected result for controlled input rows?**

For:

```sql
case
    when amount >= 1000 then 'large'
    else 'small'
end
```

a unit test can supply `999` and `1000` and assert the two expected classifications.

Unit tests are especially useful for complex logic, edge cases, and refactors.

Reference: https://docs.getdbt.com/docs/build/unit-tests

In [ ]:
# Create a small model whose business logic is a good candidate for a unit test.

(PROJECT / "models/marts/order_size_band.sql").write_text("""
select
    order_id,
    order_amount,
    case
        when order_amount >= 1000 then 'large'
        else 'small'
    end as order_size_band
from {{ ref('stg_orders') }}
""".strip() + "\n")

In [ ]:
# Define a unit test beside the model under model-paths.
# Unit tests use static fixture rows to validate transformation logic.

(PROJECT / "models/unit_tests.yml").write_text("""
version: 2

unit_tests:
  - name: order_size_band_logic
    description: "Large orders are classified at 1000 or above."
    model: order_size_band
    given:
      - input: ref('stg_orders')
        rows:
          - {order_id: 1, customer_id: 1, order_date: '2025-01-01', status: 'completed', order_amount: 999}
          - {order_id: 2, customer_id: 1, order_date: '2025-01-01', status: 'completed', order_amount: 1000}
    expect:
      rows:
        - {order_id: 1, order_amount: 999, order_size_band: 'small'}
        - {order_id: 2, order_amount: 1000, order_size_band: 'large'}
""".strip() + "\n")

unit_result = run_dbt("test", "--select", "order_size_band_logic", check=False)
print("unit test exit code:", unit_result.returncode)

## Choosing the right test

| Question | Test |
|---|---|
| Is this column unique? | data test |
| Does every customer_id exist? | data test |
| Did this CASE expression handle edge cases? | unit test |
| Did today's source contain duplicates? | data test |
| Did a refactor change transformation behavior? | unit test |

Unit tests validate logic. Data tests validate actual data.

# 21. Model contracts

A contract describes the expected structural interface of a model.

Think of it as an API schema:

```text
fct_orders
├── order_id       integer
├── customer_id    integer
├── order_date     date
├── status         varchar
└── order_amount   decimal
```

With an enforced contract, dbt can validate that the model output conforms to the declared structure.

### Contract vs data test

```text
contract
  = expected structure/interface

data test
  = expected data behavior
```

Contracts are particularly useful for important, highly reused models.

In [ ]:
# Create a contracted model with explicit output types.
# Adapter-level physical constraints remain a separate concern.

(PROJECT / "models/marts/fct_orders_contract.sql").write_text("""
select
    cast(order_id as integer) as order_id,
    cast(customer_id as integer) as customer_id,
    cast(order_date as date) as order_date,
    cast(status as varchar) as status,
    cast(order_amount as decimal(12,2)) as order_amount
from {{ ref('fct_orders') }}
""".strip() + "\n")

(PROJECT / "models/marts/contracts.yml").write_text("""
version: 2

models:
  - name: fct_orders_contract
    description: "Contracted interface for analytical orders."
    config:
      contract:
        enforced: true
    columns:
      - name: order_id
        data_type: integer
      - name: customer_id
        data_type: integer
      - name: order_date
        data_type: date
      - name: status
        data_type: varchar
      - name: order_amount
        data_type: decimal(12,2)
""".strip() + "\n")

contract_result = run_dbt("build", "--select", "fct_orders_contract", check=False)
print("contract build exit code:", contract_result.returncode)

## Contract caveat

Contract enforcement and physical database constraints are adapter-specific.

Do not assume:

```text
dbt contract = physical primary key / foreign key enforcement
```

A contract primarily describes the model interface. Physical constraint enforcement depends on the target platform and adapter.

Reference: https://docs.getdbt.com/reference/resource-properties/constraints

# 22. Incremental models

Imagine:

```text
1 billion historical rows
+
10 million new/changed rows every day
```

A full rebuild every day may be unnecessarily expensive.

An incremental model says:

> On the first run, build everything. On later runs, process only the rows that should be processed.

Core configuration:

```jinja
{{ config(
    materialized='incremental',
    unique_key='event_id'
) }}
```

and:

```jinja
{% if is_incremental() %}
where ...
{% endif %}
```

`is_incremental()` is true when the target already exists, the model is incremental, and `--full-refresh` was not requested.

Reference: https://docs.getdbt.com/docs/build/incremental-models

In [ ]:
# Create an incremental event model.
# The filter controls what is processed on incremental runs.

(PROJECT / "models/marts/fct_events.sql").write_text("""
{{
    config(
        materialized='incremental',
        unique_key='event_id'
    )
}}

select
    event_id,
    customer_id,
    cast(event_at as timestamp) as event_at,
    event_type
from {{ ref('stg_events') }}

{% if is_incremental() %}
where cast(event_at as timestamp) >= (
    select coalesce(max(event_at), timestamp '1900-01-01')
    from {{ this }}
)
{% endif %}
""".strip() + "\n")

run_dbt("run", "--select", "fct_events")
print(duck_query("select count(*) as rows, max(event_at) as max_event_at from fct_events"))

In [ ]:
# Simulate a later batch and rerun the incremental model.

events2 = pd.concat([
    events,
    pd.DataFrame([
        [6, 4, "2025-03-03 09:00:00", "view"],
        [7, 5, "2025-03-03 10:00:00", "purchase"],
    ], columns=events.columns),
], ignore_index=True)

events2.to_csv(PROJECT / "seeds/raw_events.csv", index=False)

run_dbt("seed", "--select", "raw_events")
run_dbt("run", "--select", "fct_events")

print(duck_query("select * from fct_events order by event_id"))

# 23. Incremental correctness: late-arriving data

Incremental is not automatically "better".

You must define:

### What is new?

Usually a timestamp watermark.

### What is changed?

Often an `updated_at` column plus a stable unique key.

### What about late data?

Example:

```text
event timestamp = Day 8
ingestion timestamp = Day 10
```

A strict:

```sql
event_at > max(event_at)
```

filter can miss it.

### Production pattern

For important incremental models, explicitly define:

```text
watermark
+
lookback window
+
deduplication
+
unique key
+
late-data policy
+
full-refresh/recovery procedure
```

Incremental processing is therefore a **correctness design problem**, not merely a performance trick.

In [ ]:
# Demonstrate a lookback-window pattern.
# Reprocessing a recent window is one common strategy for handling late-arriving events.

(PROJECT / "models/marts/fct_events_lookback.sql").write_text("""
{{
    config(
        materialized='incremental',
        unique_key='event_id'
    )
}}

select
    event_id,
    customer_id,
    cast(event_at as timestamp) as event_at,
    event_type
from {{ ref('stg_events') }}

{% if is_incremental() %}
where cast(event_at as timestamp) >= (
    select
        coalesce(max(event_at), timestamp '1900-01-01') - interval 1 day
    from {{ this }}
)
{% endif %}
""".strip() + "\n")

# 24. Snapshots: preserving history

An incremental model answers:

> **How can I efficiently update the current target?**

A snapshot answers:

> **How did this record change over time?**

Example history:

```text
2025-01-10  Asha  Delhi
2025-04-10  Asha  Mumbai
```

A snapshot can preserve both states with validity metadata.

Useful for:

- customer attributes,
- account status,
- slowly changing dimensions,
- historical configuration.

Choose the snapshot strategy according to the source's change semantics.

In [ ]:
# Create a snapshot using the check strategy.
# The snapshot watches customer_name and city for changes.

(PROJECT / "snapshots/customers_snapshot.sql").write_text("""
{% snapshot customers_snapshot %}

{{
    config(
        target_schema='snapshots',
        unique_key='customer_id',
        strategy='check',
        check_cols=['customer_name', 'city']
    )
}}

select
    customer_id,
    customer_name,
    city,
    signup_date
from {{ ref('stg_customers') }}

{% endsnapshot %}
""".strip() + "\n")

run_dbt("snapshot")

In [ ]:
# Simulate a customer attribute change and capture another snapshot version.
# This demonstrates why snapshots are different from current-state models.

customers_changed = customers.copy()
customers_changed.loc[customers_changed["customer_id"] == 1, "city"] = "Mumbai"
customers_changed.to_csv(PROJECT / "seeds/raw_customers.csv", index=False)

run_dbt("seed", "--select", "raw_customers")
run_dbt("run", "--select", "stg_customers")
run_dbt("snapshot")

print(duck_query("""
select *
from snapshots.customers_snapshot
where customer_id = 1
order by dbt_valid_from
"""))

# 25. Source freshness

Data quality has two dimensions:

```text
Is the data correct?
        +
Is the data arriving on time?
```

A transformation can be correct while the pipeline is operationally wrong because the source is stale.

A freshness policy can express expectations such as:

```yaml
config:
  freshness:
    warn_after: {count: 1, period: day}
    error_after: {count: 3, period: day}
loaded_at_field: _etl_loaded_at
```

The exact command and configuration surface depends on the dbt line. In dbt v1.12, the source freshness workflow remains available as `dbt source freshness`; newer dbt lines also provide a broader `dbt freshness` command.

Reference: https://docs.getdbt.com/reference/resource-configs/freshness

In [ ]:
# Add a synthetic ingestion timestamp so freshness can be demonstrated without an external source.

customers_fresh = customers.copy()
customers_fresh["_etl_loaded_at"] = "2025-03-18 12:00:00"
customers_fresh.to_csv(PROJECT / "seeds/raw_customers.csv", index=False)

(PROJECT / "models/staging/sources.yml").write_text("""
version: 2

sources:
  - name: acme_raw
    schema: main
    config:
      freshness:
        warn_after: {count: 1, period: day}
        error_after: {count: 3, period: day}
      loaded_at_field: _etl_loaded_at
    tables:
      - name: raw_customers
      - name: raw_orders
      - name: raw_order_items
      - name: raw_payments
      - name: raw_events
""".strip() + "\n")

run_dbt("seed", "--select", "raw_customers")
freshness_result = run_dbt("source", "freshness", check=False)
print("freshness exit code:", freshness_result.returncode)

# 26. Documentation

A useful dbt project documents:

- models,
- columns,
- sources,
- tests,
- exposures,
- ownership,
- business meaning.

Good documentation explains:

```text
What is this model?
What is its grain?
What does the important column mean?
What assumptions does it make?
Who owns it?
Who consumes it?
```

Bad documentation merely repeats names.

Documentation is most valuable when it captures **meaning and assumptions** that are not obvious from SQL.

In [ ]:
# Add richer business documentation to the core mart.

(PROJECT / "models/marts/documentation.yml").write_text("""
version: 2

models:
  - name: fct_orders
    description: >
      Curated order-level fact table. Each row represents one non-cancelled
      customer order and is intended for revenue and order analytics.
    columns:
      - name: order_id
        description: "Unique identifier for the commerce order."
      - name: customer_id
        description: "Identifier of the customer who placed the order."
      - name: order_date
        description: "Business date on which the order was placed."
      - name: status
        description: "Standardized order lifecycle status."
      - name: order_amount
        description: "Order value represented as a decimal monetary amount."
""".strip() + "\n")

run_dbt("docs", "generate", check=False)

# 27. Exposures

An **exposure** represents a downstream use of the dbt project:

- dashboard,
- application,
- notebook,
- ML pipeline,
- report.

Without exposures:

```text
model → ??? → dashboard
```

With an exposure:

```text
model → executive_sales_report
```

This makes downstream impact visible.

Reference: https://docs.getdbt.com/docs/build/exposures

In [ ]:
# Declare a downstream dashboard-like exposure.
# It connects the governed model graph to a consumer.

(PROJECT / "models/exposures.yml").write_text("""
version: 2

exposures:
  - name: executive_sales_report
    label: Executive Sales Report
    type: dashboard
    maturity: medium
    description: "Illustrative executive sales dashboard."
    depends_on:
      - ref('fct_orders')
    owner:
      name: "Analytics Team"
""".strip() + "\n")

run_dbt("parse")

# 28. Artifacts and metadata

Important generated artifacts include:

### `manifest.json`

A machine-readable representation of the project's resources and graph.

Useful for:

- lineage,
- dependency analysis,
- metadata automation,
- state comparison,
- project introspection.

### `run_results.json`

Execution information.

Useful for:

- runtime analysis,
- pass/fail monitoring,
- operational reporting.

### `catalog.json`

Database metadata used by documentation/catalog workflows.

The key lesson:

> **dbt produces metadata about the transformation system, not just database tables.**

Reference: https://docs.getdbt.com/reference/artifacts/dbt-artifacts

In [ ]:
# Generate and inspect the manifest at a high level.
# We inspect summary metadata rather than dumping the full JSON into the notebook.

run_dbt("parse")

target = PROJECT / "target"
print("Target JSON artifacts:")
for p in sorted(target.glob("*.json")):
    print(" -", p.name)

manifest_path = target / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    nodes = manifest.get("nodes", {})
    print("\nManifest summary:")
    print("models:", sum(v.get("resource_type") == "model" for v in nodes.values()))
    print("tests:", sum(v.get("resource_type") == "test" for v in nodes.values()))
    print("seeds:", sum(v.get("resource_type") == "seed" for v in nodes.values()))
    print("snapshots:", sum(v.get("resource_type") == "snapshot" for v in nodes.values()))
    print("sources:", len(manifest.get("sources", {})))
    print("exposures:", len(manifest.get("exposures", {})))

# 29. Selection and graph traversal

Once you have a DAG, you rarely want to rebuild everything.

For:

```text
A → B → C
```

conceptually:

```text
B       = B
+B      = ancestors + B
B+      = B + descendants
+B+     = ancestors + B + descendants
```

Selection can also use:

- names,
- paths,
- tags,
- resource types,
- state,
- sources.

This is where the DAG becomes an operational tool.

In [ ]:
# Demonstrate graph-aware selection.
# dbt ls only lists resources, so these examples do not modify the database.

run_dbt("ls", "--select", "+fct_orders")
run_dbt("ls", "--select", "fct_orders+")
run_dbt("ls", "--select", "tag:staging")

# 30. `compile` vs `run` vs `test` vs `build`

### `dbt compile`

Generate compiled SQL without executing model SQL.

### `dbt run`

Build models.

### `dbt test`

Run tests.

### `dbt build`

Build and test resources in DAG order and is often the most useful integrated workflow for development and CI.

A practical loop:

```text
edit
 ↓
compile
 ↓
inspect SQL
 ↓
build impacted graph
 ↓
inspect data
 ↓
test
```

Think in terms of **compile correctness + execution correctness + data correctness**.

In [ ]:
# Build a focused graph around the core fact.
# This demonstrates a repeatable development workflow.

run_dbt("compile", "--select", "fct_orders")
run_dbt("build", "--select", "+fct_orders")

# 31. Debugging like a data engineer

When dbt fails:

## 1. Classify the failure

```text
installation?
connection?
parsing?
compilation?
SQL execution?
data test?
environment?
concurrency?
```

## 2. Read the first meaningful dbt error

A long traceback may contain symptoms after the root cause.

## 3. Compile

```text
dbt compile --select model_name
```

## 4. Inspect generated SQL

Jinja problems often become obvious after compilation.

## 5. Inspect upstream relations

Check:

- schema,
- types,
- row counts,
- nulls,
- duplicate keys.

## 6. Inspect graph

```text
dbt ls --select +model_name
```

## 7. Inspect artifacts

`manifest.json` tells you what dbt parsed. `run_results.json` tells you what happened during execution.

### The key debugging principle

> **Separate "my transformation is wrong" from "my environment is wrong."**

The original DuckDB lock error was an environment/resource-lifecycle failure, not a transformation failure.

In [ ]:
# Inspect compiled SQL and recent run results.
# This is the evidence-driven debugging workflow learners should practice.

run_dbt("compile", "--select", "fct_orders")

compiled_files = list(
    (PROJECT / "target" / "compiled" / "dbt_colab_lab").rglob("fct_orders.sql")
)
for p in compiled_files:
    print(f"--- {p} ---")
    print(p.read_text())

run_results_path = PROJECT / "target" / "run_results.json"
if run_results_path.exists():
    run_results = json.loads(run_results_path.read_text())
    print("\nRecent statuses:")
    for item in run_results.get("results", [])[-10:]:
        print(item.get("unique_id"), "=>", item.get("status"))

# 32. Hooks

Hooks can run SQL around dbt lifecycle events:

- pre-hook,
- post-hook,
- on-run-start,
- on-run-end.

Useful for:

- auditing,
- session setup,
- controlled maintenance,
- lifecycle-specific operations.

But hooks introduce side effects outside the model's normal `SELECT`.

Production rule:

> **If a hook is important to correctness, document and test it like application code.**

In [ ]:
# Demonstrate a harmless post-hook that creates an audit marker.
# The example is intentionally simple so the learner can focus on lifecycle behavior.

(PROJECT / "models/marts/hook_demo.sql").write_text("""
{{
    config(
        materialized='table',
        post_hook="create table if not exists dbt_lab_audit as select 'hook_demo' as model_name"
    )
}}

select
    1 as id,
    'hook example' as message
""".strip() + "\n")

hook_result = run_dbt("run", "--select", "hook_demo", check=False)
print("hook demo exit code:", hook_result.returncode)

# 33. Analyses

An **analysis** is SQL stored and version-controlled in the dbt project without being a normal persistent model.

Useful for:

- investigations,
- stakeholder analysis,
- exploratory work worth reviewing,
- important one-off SQL.

Think:

```text
model
  = production dataset

analysis
  = version-controlled analytical query
```

In [ ]:
# Create an analysis that consumes the governed order fact.

(PROJECT / "analyses/revenue_by_city.sql").write_text("""
select
    city,
    sum(order_amount) as revenue
from {{ ref('fct_orders') }}
group by city
order by revenue desc
""".strip() + "\n")

run_dbt("compile", "--select", "revenue_by_city")

# 34. Packages and `dbt_utils`

Packages distribute reusable dbt code.

The canonical example is `dbt_utils`.

Benefits:

- reusable macros,
- reusable generic tests,
- standardized patterns,
- less duplicated project code.

Workflow:

```text
packages.yml
    ↓
dbt deps
    ↓
package installed
    ↓
macros/tests available
```

### Engineering rule

Before writing your own utility macro, ask:

> "Does a maintained package already provide this?"

Pin package versions in production for reproducibility.

Reference: https://hub.getdbt.com/dbt-labs/dbt_utils/latest/

In [ ]:
# Declare a pinned dbt-utils dependency.
# The dependency is deliberately explicit so the project is reproducible.

(PROJECT / "packages.yml").write_text("""
packages:
  - package: dbt-labs/dbt_utils
    version: 1.4.1
""".strip() + "\n")

deps_result = run_dbt("deps", check=False)
print("dbt deps exit code:", deps_result.returncode)

# 35. Python models with dbt-duckdb

dbt is SQL-first, but dbt-duckdb supports Python models.

Conceptually:

```text
SQL model
    ↓
SQL + Jinja
    ↓
database relation

Python model
    ↓
Python function
    ↓
DataFrame / Relation
    ↓
database relation
```

Use Python when the transformation genuinely benefits from Python libraries or algorithms that are awkward in SQL.

If a transformation is equally clear and performant in SQL, SQL is generally the more portable choice.

Reference: https://github.com/duckdb/dbt-duckdb

In [ ]:
# Create a small Python model using the dbt-duckdb Python-model interface.
# It reads a dbt relation and returns a pandas DataFrame.

(PROJECT / "models/marts/customer_python_model.py").write_text("""
def model(dbt, session):
    dbt.config(materialized="table")

    df = dbt.ref("stg_customers").df()

    df["customer_name_upper"] = df["customer_name"].str.upper()

    return df[
        ["customer_id", "customer_name", "city", "customer_name_upper"]
    ]
""".strip() + "\n")

python_result = run_dbt("run", "--select", "customer_python_model", check=False)
print("Python model exit code:", python_result.returncode)

# 36. Model versions

As a platform matures, important models become interfaces.

A breaking change can be introduced as a new version:

```text
fct_orders v1
    └── existing consumers

fct_orders v2
    └── new consumers
```

This is analogous to API versioning.

### Why it matters

Without versioning:

```text
breaking change
      ↓
all consumers break
```

With versioning:

```text
new version
      ↓
migration window
      ↓
consumers move
      ↓
old version retired
```

Model versions are especially useful for mature, high-value interfaces.

Reference: https://docs.getdbt.com/docs/mesh/govern/model-versions

In [ ]:
# Create two versions of a customer-value model to illustrate interface evolution.

(PROJECT / "models/marts/customer_value_v1.sql").write_text("""
select
    customer_id,
    sum(order_amount) as customer_value
from {{ ref('fct_orders') }}
group by customer_id
""".strip() + "\n")

(PROJECT / "models/marts/customer_value_v2.sql").write_text("""
select
    customer_id,
    sum(order_amount) as lifetime_value,
    count(*) as order_count
from {{ ref('fct_orders') }}
group by customer_id
""".strip() + "\n")

(PROJECT / "models/marts/model_versions.yml").write_text("""
version: 2

models:
  - name: customer_value
    latest_version: 2
    versions:
      - v: 1
        defined_in: customer_value_v1.sql
      - v: 2
        defined_in: customer_value_v2.sql
""".strip() + "\n")

version_result = run_dbt("parse", check=False)
print("version parse exit code:", version_result.returncode)

# 37. State-aware CI/CD

A mature CI workflow should avoid rebuilding everything for every pull request.

Conceptually:

```text
pull request
     ↓
compare current project with previous state
     ↓
identify modified resources
     ↓
build affected graph
     ↓
run tests
     ↓
review
     ↓
merge
```

Common ideas include:

- state comparison,
- `state:modified`,
- slim CI,
- deferred references,
- production artifacts,
- selective graph builds.

The exact implementation depends on the dbt release and deployment environment.

The principle is:

> **CI should validate the impact of a change, not blindly rebuild the entire data platform.**

# 38. Production project architecture

A mature project might look like:

```text
models/
├── staging/
│   ├── shopify/
│   ├── payments/
│   └── marketing/
│
├── intermediate/
│   ├── orders/
│   ├── customers/
│   └── finance/
│
└── marts/
    ├── finance/
    ├── marketing/
    ├── product/
    └── customer/
```

Around it:

```text
macros/
tests/
snapshots/
seeds/
analyses/
packages.yml
dbt_project.yml
```

## Production design principles

1. **Stable grain** — every model has a clear grain.
2. **Explicit ownership** — important models have owners.
3. **Tests at boundaries** — protect important assumptions.
4. **Contracts for critical interfaces** — make structure explicit.
5. **Incremental only when justified** — use it for scale/cost.
6. **Documentation as code** — meaning travels with transformations.
7. **Small composable models** — avoid giant opaque SQL.
8. **Minimal hidden side effects** — use hooks carefully.
9. **Reproducible dependencies** — pin versions.
10. **CI protects production** — compile, unit-test where useful, build impacted graph, test data.

# 39. Capstone — build a production-style Acme Mart

Now stop following cells mechanically.

## Sources

```text
customers
orders
order_items
payments
events
```

## Staging

```text
stg_customers
stg_orders
stg_order_items
stg_payments
stg_events
```

## Intermediate

```text
int_orders_enriched
int_customer_order_metrics
int_payment_status
```

## Marts

```text
fct_orders
fct_events
dim_customers
customer_order_summary
```

## Requirements

Include:

- `source()` for upstream relations,
- `ref()` between dbt models,
- at least one seed,
- staging/intermediate/mart layers,
- built-in data tests,
- one singular test,
- one generic test,
- one macro,
- one Jinja-driven model,
- one configurable variable,
- one incremental model,
- one snapshot,
- model documentation,
- one exposure,
- artifact inspection,
- selective graph execution,
- at least one unit test,
- one contract on a critical model.

## Design questions

1. What is the grain of every mart?
2. Which sources feed each mart?
3. Which tests protect each assumption?
4. Which transformations are expensive?
5. Which models should be incremental?
6. What happens when data arrives late?
7. How do you recover from a bad incremental run?
8. Which models are public interfaces?
9. Which dashboards depend on them?
10. Which changes require versioning?
11. How would CI validate a pull request?
12. What would you monitor in production?

# 40. Final knowledge map

```text
                         dbt project
                              │
            ┌─────────────────┼─────────────────┐
            │                 │                 │
         sources            seeds            packages
            │
            ▼
        source()
            │
            ▼
        staging
            │
           ref()
            │
            ▼
      intermediate
            │
           ref()
            │
            ▼
          marts
            │
     ┌──────┼────────┐
     │      │        │
   tests  docs   exposures
     │      │        │
     └──────┼────────┘
            │
         artifacts
            │
      CI / governance
```

Around the graph:

```text
Jinja → macros → reusable SQL generation

incremental → efficient processing

snapshots → historical state

contracts → structural guarantees

unit tests → transformation logic correctness

source freshness → operational timeliness

model versions → safe interface evolution
```

## The important mental shift

The beginner asks:

> "How do I write this SQL?"

The analytics engineer asks:

> "What is the data contract, grain, dependency, ownership, quality guarantee, execution strategy, and consumer impact of this transformation?"

That shift is the real skill.

# 41. Final practical checklist

### Semantics
- [ ] Is the model grain explicit?
- [ ] Are business definitions documented?
- [ ] Are joins preserving intended grain?

### Dependencies
- [ ] Are dbt models referenced with `ref()`?
- [ ] Are external tables declared as `source()`?
- [ ] Is lineage visible?

### Quality
- [ ] Are keys tested?
- [ ] Are relationships tested?
- [ ] Are business rules tested?
- [ ] Do complex transformations have unit tests?

### Performance
- [ ] Is the materialization appropriate?
- [ ] Is incremental processing justified?
- [ ] Is the incremental filter correct for late data?

### Governance
- [ ] Is ownership documented?
- [ ] Are critical interfaces contracted?
- [ ] Are downstream exposures known?
- [ ] Are breaking changes versioned?

### Operations
- [ ] Can the model be selected independently?
- [ ] Can it be debugged from compiled SQL?
- [ ] Are artifacts available?
- [ ] Is there a CI path?

### Reproducibility
- [ ] Are dbt and adapter versions controlled?
- [ ] Are package versions pinned?
- [ ] Are secrets outside source control?

# Official references

- dbt v1.12: https://docs.getdbt.com/docs/dbt-versions/upgrading-to-v1.12
- Models: https://docs.getdbt.com/docs/build/models
- `ref()`: https://docs.getdbt.com/reference/dbt-jinja-functions/ref
- Sources: https://docs.getdbt.com/docs/build/sources
- Data tests: https://docs.getdbt.com/docs/build/data-tests
- Jinja/macros: https://docs.getdbt.com/docs/build/jinja-macros
- Incremental models: https://docs.getdbt.com/docs/build/incremental-models
- Unit tests: https://docs.getdbt.com/docs/build/unit-tests
- Exposures: https://docs.getdbt.com/docs/build/exposures
- Model versions: https://docs.getdbt.com/docs/mesh/govern/model-versions
- Artifacts: https://docs.getdbt.com/reference/artifacts/dbt-artifacts
- Freshness: https://docs.getdbt.com/reference/resource-configs/freshness
- DuckDB concurrency: https://duckdb.org/docs/current/connect/concurrency
- dbt-duckdb: https://github.com/duckdb/dbt-duckdb
- dbt-utils: https://hub.getdbt.com/dbt-labs/dbt_utils/latest/

# Instructor note — what was fixed from the previous notebook

The previous notebook was a useful feature survey, but several structural issues prevented it from being a reliable world-class lab.

### 1. DuckDB connection lifetime
A Python `duckdb.connect()` handle remained open while dbt tried to write the same `.duckdb` file. This caused the observed file-lock error.

**Fix:** the new notebook uses short-lived query connections and explicitly closes them before every dbt command.

### 2. Global package installation
Installing dbt directly into Colab's global environment caused unrelated dependency conflicts.

**Fix:** dbt runs inside a dedicated virtual environment.

### 3. Configuration hierarchy
The previous project configured `models/staging`, `intermediate`, and `marts`, but initially placed `hello.sql` directly under `models/`, causing unused configuration warnings.

**Fix:** models are placed under the configured directories from the beginning.

### 4. Teaching flow
The old notebook introduced many features but did not consistently establish the theory before the practical.

**Fix:** every major concept now follows theory → implementation → inspection → debugging/experiment → production reasoning.

### 5. Version awareness
The new notebook explicitly pins and prints the dbt/adapter versions.

### 6. Failure-aware teaching
The notebook treats environment failures, compilation failures, execution failures, and data-quality failures as different classes of problems.

In [ ]:
# FINAL PROJECT CHECK
# This cell performs a lightweight structural verification.
# It does not keep a DuckDB connection open after execution.

required_paths = [
    "dbt_project.yml",
    "profiles.yml",
    "models/staging",
    "models/intermediate",
    "models/marts",
    "seeds",
    "snapshots",
    "tests",
    "macros",
    "analyses",
]

missing = [p for p in required_paths if not (PROJECT / p).exists()]
print("Missing required paths:", missing)

parse_result = run_dbt("parse", check=False)
print("Final parse exit code:", parse_result.returncode)

if not missing and parse_result.returncode == 0:
    print("Project structure and dbt parsing: OK")
else:
    print("Review the parse output before treating the project as runnable.")